# 🥇 05 — Dhara: Gold Layer (ML-Ready + Analytics Views)

Creates the final feature table that Drishti's GBT model will train on, plus analytics views for the AI/BI dashboard and Genie space.

**Key Databricks depth:** `OPTIMIZE ... ZORDER BY` on the Gold feature table for fast lookups during real-time prediction.

In [0]:
%sql
USE CATALOG setu_rail;
USE SCHEMA gold;

## Analytics views (used by the dashboard + Genie)

In [0]:
%sql
CREATE OR REPLACE VIEW setu_rail.gold.vw_trains_by_city AS
SELECT city,
       COUNT(*)                          AS scheduled_trains,
       ROUND(AVG(pm25_avg), 1)           AS avg_pm25,
       ROUND(AVG(no2_avg), 1)            AS avg_no2
FROM   setu_rail.silver.train_enriched
GROUP BY city;

CREATE OR REPLACE VIEW setu_rail.gold.vw_hourly_schedule AS
SELECT scheduled_hour,
       COUNT(*) AS num_trains
FROM   setu_rail.silver.train_enriched
WHERE  scheduled_hour IS NOT NULL
GROUP BY scheduled_hour
ORDER BY scheduled_hour;

CREATE OR REPLACE VIEW setu_rail.gold.vw_rules_by_source AS
SELECT source, source_title, COUNT(*) AS chunk_count,
       MIN(page) AS first_page, MAX(page) AS last_page
FROM   setu_rail.silver.rules_chunks
GROUP BY source, source_title;

CREATE OR REPLACE VIEW setu_rail.gold.vw_top_polluted_routes AS
SELECT city,
       ROUND(AVG(pm25_avg), 1) AS avg_pm25,
       COUNT(DISTINCT train_no) AS trains_affected
FROM   setu_rail.silver.train_enriched
WHERE  pm25_avg IS NOT NULL
GROUP BY city
ORDER BY avg_pm25 DESC;

In [0]:
%sql
-- Quick sanity check all views return rows
SELECT 'vw_trains_by_city'        AS v, COUNT(*) AS n FROM setu_rail.gold.vw_trains_by_city
UNION ALL SELECT 'vw_hourly_schedule',       COUNT(*) FROM setu_rail.gold.vw_hourly_schedule
UNION ALL SELECT 'vw_rules_by_source',       COUNT(*) FROM setu_rail.gold.vw_rules_by_source
UNION ALL SELECT 'vw_top_polluted_routes',   COUNT(*) FROM setu_rail.gold.vw_top_polluted_routes;

✅ Feature table itself is built in `02_drishti_ml_pipeline/01_synthesize_delays.ipynb` because it needs the label column.

**Next:** `02_drishti_ml_pipeline/01_synthesize_delays.ipynb`